In [1]:
import numpy as np
import scipy
import scipy.special as sp
import matplotlib.pyplot as plt
import h5py
from localized_2d import WfnParams, Wavefunction, HamiltParams, Hamilt

In [6]:
def get_eff_mass_for_coupling(dir,Mx,My,tx,ty,B,V_0):
    e_arr = np.zeros(len(V_0_arr), dtype=complex)
    e_arr_tun = np.zeros(len(V_0_arr), dtype=complex)
    
    e_T_arr = np.zeros(len(V_0_arr), dtype=complex)
    e_V_arr = np.zeros(len(V_0_arr), dtype=complex)
    e_B_arr = np.zeros(len(V_0_arr), dtype=complex)

    Tx_arr = np.zeros(len(V_0_arr), dtype=complex)
    Ty_arr = np.zeros(len(V_0_arr), dtype=complex)

    file_path = dir+'data_B_'+str(B)+'_V0_'+str(V_0)+'_tx_'+str(tx)+'_ty_'+str(ty)+'_'+str(int(Mx))+'x'+str(int(My))+'.h5'
    
    with h5py.File(file_path, 'r') as f:
        phi_data = f["wfn"][...]
        Mx = f["Mx"][...]
        My = f["My"][...]
        n = f["n"][...]
        B = f["B"][...]
        tx = f["tx"][...]
        ty = f["ty"][...]
        V_0 = f["V0"][...]
        e_tun = f["e_tun"][...]
        v_tun = f["v_tun"][...]
        
    wfnpar = WfnParams(Mx, My, n)
    Mx, My, n, x, k2 = wfnpar.read()

    wfn = Wavefunction(wfnpar, array=phi_data)

    hamiltpar = HamiltParams(B, tx, ty, V_0)

    H = Hamilt(wfnpar, hamiltpar)
    energy_tot, energy_rot, energy_ele, energy_int = H.calculate_energy(wfn)
    Tx, Ty = H.calc_Tx_Ty(wfn)

    H, B = H.tunneling_hamiltonian(wfn)

    '''
    Here starts computation of matrix elements
    '''
    get_matrix = lambda fun_: np.array([[fun_(m, n) for n in range(My)] for m in range(Mx)])
    elewfn_shift_mat = get_matrix(wfn.elewfn_shift)
    rotwfn_shift_mat = get_matrix(wfn.rotwfn_shift)

    Oe = np.einsum('mnij,ij->mnij', np.conj(elewfn_shift_mat), wfn.elewfn)
    OE = np.sum(Oe, axis=(2, 3))

    Or = np.einsum('mnkij,kij->mnij', np.conj(rotwfn_shift_mat), wfn.rotwfn)
    OR = np.prod(Or, axis=(2, 3))

    He = (-tx * (np.roll(OE, 1, axis=0) + np.roll(OE, -1, axis=0)) \
        -ty * (np.roll(OE, 1, axis=1) + np.roll(OE, -1, axis=1))) * OR

    return He


'''
Get State Properties for a phase diagram scan in the delta_t-V_0 plane
'''
V_0_arr = np.array([0.0,0.6,1.0])

Mx = 20
My = 20
tx = 0.5
ty = 0.5
B = 1e-2

mx = np.zeros(len(V_0_arr))
my = np.zeros(len(V_0_arr))
dir = 'data_localized/'
for i in range(len(V_0_arr)):
    He = get_eff_mass_for_coupling(dir=dir,Mx=Mx,My=My,tx=tx,ty=ty,B=B,V_0=V_0_arr[i])

    indx = np.repeat(np.arange(Mx)[::-1]+1, (Mx,)).reshape(My,Mx)
    multiplicities = indx*indx.T
    
    kx = np.arange(Mx)
    kx[kx > Mx/2] = -(kx[kx > Mx/2]%(Mx/2))[::-1]

    ky = np.arange(My)
    ky[ky > My/2] = -(ky[ky > My/2]%(My/2))[::-1]

    mx2Hem = np.einsum('ij,i->ij', -He*multiplicities, kx**2).real
    my2Hem = np.einsum('ij,j->ij', -He*multiplicities, ky**2).real
    
    mx[i] = (Mx*My)/np.sum(mx2Hem)
    my[i] = (Mx*My)/np.sum(my2Hem)

    print('V_0, m_x, m_y =', V_0_arr[i], mx[i], my[i])

V_0, m_x, m_y = 0.0 0.00014214641417131463 0.00014214641417131463
V_0, m_x, m_y = 0.6 3780.0042092955728 3780.0042092955414
V_0, m_x, m_y = 1.0 46441.030872651885 46441.03087265275
